In [ ]:
%%writefile /kaggle/working/train_final.py
"""
Phase 2 - full training of one configuration (one process = one GPU).
Full 70% training split, early stopping on validation loss (as in Bahaa et al. 2026),
checkpoint of the best epoch, optional single evaluation on the test split.
Launched by the next notebook cell.
"""
import argparse, glob, json, math, os, random, time

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

INPUT_ROOT = "/kaggle/input"
OUT_DIR = "/kaggle/working"
IMG_SIZE = 224


def find_image_root(sample_path):
    if os.path.exists(os.path.join(INPUT_ROOT, sample_path)):
        return INPUT_ROOT
    first = sample_path.split("/")[0]
    for d in glob.glob(f"{INPUT_ROOT}/**/{first}", recursive=True):
        root = os.path.dirname(d)
        if os.path.exists(os.path.join(root, sample_path)):
            return root
    raise FileNotFoundError(f"Could not find images for {sample_path} - attach the PlantVillage dataset.")


class LeafDataset(Dataset):
    def __init__(self, df, tf, root):
        self.paths = [os.path.join(root, p) for p in df["path"]]
        self.labels = df["label_idx"].tolist()
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


def build_transforms(model_name):
    # Same augmentation as Phase 1; normalisation = the model's own pretraining statistics
    # (ImageNet mean/std for MobileNetV3, [0, 1] scaling for MobileViT) - identical to Phase 1.
    cfg = timm.models.get_pretrained_cfg(model_name).to_dict()
    norm = transforms.Normalize(cfg["mean"], cfg["std"])
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=(0.5, 1.5)),
        transforms.ToTensor(), norm,
    ])
    eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), norm])
    return train_tf, eval_tf


def make_loader(df, tf, root, batch_size, shuffle, workers, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(LeafDataset(df, tf, root), batch_size=batch_size, shuffle=shuffle,
                      num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
                      generator=g, drop_last=shuffle)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def build_scheduler(opt, schedule, steps_per_epoch, epochs):
    total, warmup = steps_per_epoch * epochs, steps_per_epoch  # 1 warmup epoch for cosine

    def fn(step):
        if schedule == "constant":
            return 1.0
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))

    return torch.optim.lr_scheduler.LambdaLR(opt, fn)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ce = nn.CrossEntropyLoss(reduction="sum")
    total, preds, labels = 0.0, [], []
    for x, y in loader:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.amp.autocast("cuda"):
            out = model(x)
        total += ce(out.float(), y).item()
        preds.append(out.argmax(1).cpu()); labels.append(y.cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    metrics = {"loss": total / len(labels), "acc": float((preds == labels).mean()),
               "macro_f1": float(f1_score(labels, preds, average="macro"))}
    return metrics, preds, labels


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--config", required=True, help="JSON string with the hyperparameters")
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--patience", type=int, default=5)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--tag", required=True)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--no-test", action="store_true")
    args = p.parse_args()
    cfg = json.loads(args.config)
    torch.backends.cudnn.benchmark = True
    set_seed(args.seed)

    found = glob.glob(f"{INPUT_ROOT}/**/splits.csv", recursive=True)
    assert found, "splits.csv not found - attach your splits dataset."
    df = pd.read_csv(found[0])
    num_classes = df["label_idx"].nunique()
    root = find_image_root(df["path"].iloc[0])
    train_df, val_df, test_df = (df[df.split == s] for s in ("train", "val", "test"))
    train_tf, eval_tf = build_transforms(args.model)
    print(f"[{args.tag}] GPU {torch.cuda.get_device_name(0)} | {len(train_df)} train / {len(val_df)} val"
          f" / {len(test_df)} test | cfg={cfg} | seed={args.seed} | max {args.epochs} epochs", flush=True)

    model = timm.create_model(args.model, pretrained=True, num_classes=num_classes,
                              drop_rate=cfg["dropout"]).cuda()
    train_loader = make_loader(train_df, train_tf, root, cfg["batch_size"], True, args.workers, args.seed)
    val_loader = make_loader(val_df, eval_tf, root, 128, False, args.workers, args.seed)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = build_scheduler(opt, cfg["schedule"], len(train_loader), args.epochs)
    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    ckpt = f"{OUT_DIR}/{args.tag}.pt"

    history, best_loss, best_epoch, bad, t_start = [], float("inf"), 0, 0, time.time()
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tr_loss += loss.item() * x.size(0); n += x.size(0)
        val, _, _ = evaluate(model, val_loader)
        row = {"epoch": epoch, "train_loss": tr_loss / n, **{f"val_{k}": v for k, v in val.items()},
               "lr": opt.param_groups[0]["lr"], "epoch_time_s": time.time() - t0}
        history.append(row)
        print(f"[{args.tag}] " + json.dumps(row), flush=True)

        if val["loss"] < best_loss:  # early stopping + checkpoint on validation loss (as in the paper)
            best_loss, best_epoch, bad = val["loss"], epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= args.patience:
                print(f"[{args.tag}] early stopping at epoch {epoch} (best epoch {best_epoch})", flush=True)
                break
    train_time = time.time() - t_start

    model.load_state_dict(torch.load(ckpt, map_location="cuda"))
    val, _, _ = evaluate(model, val_loader)
    result = {"tag": args.tag, "model": args.model, "cfg": cfg, "seed": args.seed,
              "epochs_max": args.epochs, "epochs_run": len(history), "best_epoch": best_epoch,
              "train_time_s": train_time, "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
              "gpu": torch.cuda.get_device_name(0), "val": val, "history": history}
    if not args.no_test:
        test_loader = make_loader(test_df, eval_tf, root, 128, False, args.workers, args.seed)
        test, preds, labels = evaluate(model, test_loader)
        result["test"] = test
        classes = df.drop_duplicates("label_idx").sort_values("label_idx")["label"].tolist()
        per_class = f1_score(labels, preds, average=None, labels=range(num_classes))
        result["test_per_class_f1"] = dict(zip(classes, map(float, per_class)))
        np.savez(f"{OUT_DIR}/{args.tag}_test_preds.npz", preds=preds, labels=labels,
                 confusion=confusion_matrix(labels, preds, labels=range(num_classes)), classes=classes)
    with open(f"{OUT_DIR}/{args.tag}.json", "w") as f:
        json.dump(result, f, indent=2)
    print(f"[{args.tag}] DONE val={val}" + (f" test={result['test']}" if "test" in result else ""), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# ---- Settings ----
MODEL = "mobilenetv3_small_100"   # or "mobilevit_xxs"
RUN_TOP3_CHECK = True             # Stage A: retrain HPO top-3 on full data (10 epochs, no test)
TOP3_EPOCHS = 10
FINAL_EPOCHS = 20                 # Stage B: 3 seeds of the winner, early stopping, test evaluation
SEEDS = [0, 1, 2]

import glob, json, os, queue, subprocess, sys, threading
import torch

n_gpus = torch.cuda.device_count()
assert n_gpus > 0, "No GPU - set Accelerator to GPU T4 x2."
WORKERS = 4 if n_gpus == 1 else 2   # Kaggle has 4 CPU cores in total

found = glob.glob(f"/kaggle/input/**/hpo_{MODEL}_top3.json", recursive=True)
assert found, f"hpo_{MODEL}_top3.json not found - add the HPO notebook's output as an input."
TOP3 = json.load(open(found[0]))
for t in TOP3:
    print(f"HPO rank {t['rank']}: trial {t['trial']}  macro-F1 {t['macro_f1']:.4f}  {t['params']}")
print(f"{n_gpus} GPU(s), {WORKERS} data workers each", flush=True)


def run_jobs(jobs):
    """Run jobs (dicts with tag/config/epochs/seed/no_test) on all GPUs, one job per GPU at a time."""
    q = queue.Queue()
    for j in jobs:
        q.put(j)
    failed = []

    def gpu_worker(g):
        while True:
            try:
                j = q.get_nowait()
            except queue.Empty:
                return
            if os.path.exists(f"/kaggle/working/{j['tag']}.json"):
                print(f"[gpu{g}] {j['tag']} already done, skipping", flush=True)
                continue
            cmd = [sys.executable, "-u", "/kaggle/working/train_final.py", "--model", MODEL,
                   "--config", json.dumps(j["config"]), "--epochs", str(j["epochs"]),
                   "--seed", str(j["seed"]), "--tag", j["tag"], "--workers", str(WORKERS)]
            if j["no_test"]:
                cmd.append("--no-test")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
            proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(f"[gpu{g}] {line}", end="", flush=True)
            if proc.wait() != 0:
                failed.append(j["tag"])

    threads = [threading.Thread(target=gpu_worker, args=(g,)) for g in range(n_gpus)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    assert not failed, f"Failed jobs: {failed} - see their log lines above."


In [ ]:
# ---- Stage A: top-3 check on full training data (validation only, test untouched) ----
if RUN_TOP3_CHECK:
    run_jobs([{"tag": f"{MODEL}_top3_rank{t['rank']}", "config": t["params"], "epochs": TOP3_EPOCHS,
               "seed": 0, "no_test": True} for t in TOP3])
    check = []
    for t in TOP3:
        r = json.load(open(f"/kaggle/working/{MODEL}_top3_rank{t['rank']}.json"))
        check.append((r["val"]["macro_f1"], t))
        print(f"rank {t['rank']} (trial {t['trial']}): HPO {t['macro_f1']:.4f} -> "
              f"full-data val macro-F1 {r['val']['macro_f1']:.4f} (best epoch {r['best_epoch']})")
    WINNER = max(check, key=lambda c: c[0])[1]
    print(f"\nWinner: HPO rank {WINNER['rank']} (trial {WINNER['trial']})"
          + ("  - same as HPO ranking" if WINNER["rank"] == 1 else "  - ranking CHANGED on full data"))
else:
    WINNER = TOP3[0]
    print("Top-3 check skipped; using HPO rank 1")
json.dump(WINNER, open(f"/kaggle/working/{MODEL}_winner.json", "w"), indent=2)


In [ ]:
# ---- Stage B: final training of the winner with 3 seeds + single test evaluation ----
run_jobs([{"tag": f"{MODEL}_final_seed{s}", "config": WINNER["params"], "epochs": FINAL_EPOCHS,
           "seed": s, "no_test": False} for s in SEEDS])


In [ ]:
# ---- Summary ----
import numpy as np, pandas as pd

rows = []
for s in SEEDS:
    r = json.load(open(f"/kaggle/working/{MODEL}_final_seed{s}.json"))
    rows.append({"seed": s, "best_epoch": r["best_epoch"], "epochs_run": r["epochs_run"],
                 "train_min": r["train_time_s"] / 60, "val_acc": r["val"]["acc"],
                 "val_macro_f1": r["val"]["macro_f1"], "test_acc": r["test"]["acc"],
                 "test_macro_f1": r["test"]["macro_f1"], "params_M": r["params_M"]})
res = pd.DataFrame(rows)
res.to_csv(f"/kaggle/working/{MODEL}_final_results.csv", index=False)
print(res.round(4).to_string(index=False))
print(f"\n{MODEL} (winner = HPO trial {WINNER['trial']}), mean ± std over {len(SEEDS)} seeds:")
for col in ["val_acc", "val_macro_f1", "test_acc", "test_macro_f1"]:
    print(f"  {col:14s} {res[col].mean():.4f} ± {res[col].std(ddof=1):.4f}")

# Classes with the lowest test F1 (averaged over seeds)
pc = pd.DataFrame([json.load(open(f"/kaggle/working/{MODEL}_final_seed{s}.json"))["test_per_class_f1"]
                   for s in SEEDS]).mean().sort_values()
print("\nLowest per-class test F1:")
print(pc.head(5).round(4).to_string())
